# Data Prep: Premier League 2015/16 incidents
**IBM Z Datathon 2026**

Builds `incidents_pl1516.csv`, the shared dataset used in `02_analysis_and_model.ipynb`.

- Source: StatsBomb Open Data (free, no login), Premier League 2015/16, all 380 matches.
- `foul`, `yellow`, `red` come from "Foul Committed" events; second yellows count as yellow (the foul itself was yellow-level, the red came from the earlier booking).
- `no_foul` comes from tackles ("Duel" events of type "Tackle") that weren't called.
- Only features that exist for **both** fouls and tackles are kept. Foul-only columns (foul type, card, advantage) would let the model learn "has a foul type = foul".
- Fixed stratified 80/20 train/test split (`random_state=42`), shared by the whole team.

In [ ]:
!pip install statsbombpy pandas scikit-learn

In [ ]:
from statsbombpy import sb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

comps = sb.competitions()
comps[comps["season_name"] == "2015/2016"][["competition_id", "season_id", "competition_name"]]

Premier League 2015/16 is `competition_id=2`, `season_id=27`.

In [ ]:
matches = sb.matches(competition_id=2, season_id=27)
print(len(matches))
matches[["match_id", "home_team", "away_team", "referee"]].head()

## Download fouls and tackles for every match (10-20 minutes)

In [ ]:
all_fouls, all_tackles = [], []
for i, mid in enumerate(matches["match_id"]):
    ev = sb.events(match_id=mid)
    f = ev[ev["type"] == "Foul Committed"].copy()
    f["match_id"] = mid
    all_fouls.append(f)
    if "duel_type" in ev.columns:
        t = ev[(ev["type"] == "Duel") & (ev["duel_type"] == "Tackle")].copy()
        t["match_id"] = mid
        all_tackles.append(t)
    if i % 50 == 0:
        print(i)

info = matches[["match_id", "referee", "home_team", "away_team"]]
fouls = pd.concat(all_fouls).merge(info, on="match_id")
tackles = pd.concat(all_tackles).merge(info, on="match_id")
fouls.to_pickle("fouls_pl1516.pkl")
tackles.to_pickle("tackles_pl1516.pkl")
print(len(fouls), len(tackles))
fouls["foul_committed_card"].value_counts(dropna=False)

Result: 9,512 fouls (8,443 no card, 1,015 yellow, 25 second yellow, 29 red) and 15,445 tackles.

## Build the labelled dataset and the train/test split

In [ ]:
f = fouls.copy()
card = f["foul_committed_card"].fillna("none")
f["label"] = np.select(
    [card == "Red Card", card.isin(["Yellow Card", "Second Yellow"])],
    ["red", "yellow"], default="foul")

t = tackles.copy()
t["label"] = "no_foul"

cols = ["id", "match_id", "minute", "second", "period", "location", "player", "position", "play_pattern",
        "under_pressure", "team", "home_team", "away_team", "referee", "label"]
df = pd.concat([f[cols], t[cols]], ignore_index=True).dropna(subset=["location"])

df["x"] = df["location"].apply(lambda l: l[0])
df["y"] = df["location"].apply(lambda l: l[1])
df["is_home"] = (df["team"] == df["home_team"]).astype(int)
df["under_pressure"] = df["under_pressure"].fillna(False).astype(int)
df = df.drop(columns=["location"])

train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
df["split"] = "train"
df.loc[test.index, "split"] = "test"

df.to_csv("incidents_pl1516.csv", index=False)
pd.crosstab(df["label"], df["split"])

24,957 incidents: no_foul 15,445 · foul 8,443 · yellow 1,040 · red 29.

Note: `under_pressure` is kept in the file but must **not** be used as a feature. It is always 1 for tackles in StatsBomb's data, so it leaks the label (see the leak check in `02_analysis_and_model.ipynb`).